In [0]:
import pandas as pd

In [0]:
df = pd.read_csv("/Volumes/workspace/default/dataset/students_dropout_academic_success.csv")

In [0]:
print(df)

In [0]:
%pip install xgboost shap scikit-learn mlflow pandas numpy scipy
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
df_raw = spark.read.csv(
    '/Volumes/workspace/default/dataset/students_dropout_academic_success.csv',
    header=True,
    inferSchema=True,
    sep=';'
)
print(f'Records: {df_raw.count()}')
print(f'Columns: {len(df_raw.columns)}')
df_raw.printSchema()

In [0]:
# -------------------------------
# 1. Imports
# -------------------------------
from pyspark.sql.functions import col, sum as spark_sum, when, isnan
from pyspark.sql.types import NumericType

# -------------------------------
# 2. Load Data
# -------------------------------
df_raw = spark.read.csv(
    "/Volumes/workspace/default/dataset/students_dropout_academic_success.csv",   # 🔁 change path
    header=True,
    inferSchema=True
)

# -------------------------------
# 3. Check Schema
# -------------------------------
df_raw.printSchema()

# -------------------------------
# 4. Display sample
# -------------------------------
display(df_raw.limit(5))

# -------------------------------
# 5. Separate column types
# -------------------------------
numeric_cols = [f.name for f in df_raw.schema.fields if isinstance(f.dataType, NumericType)]
string_cols = [f.name for f in df_raw.schema.fields if not isinstance(f.dataType, NumericType)]

# -------------------------------
# 6. Null + NaN count (SAFE)
# -------------------------------
null_counts = df_raw.select([
    spark_sum(
        when(
            col(c).isNull() | (isnan(col(c)) if c in numeric_cols else False),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_raw.columns
])

display(null_counts)

# -------------------------------
# 7. Target distribution
# -------------------------------
df_raw.groupBy("target") \
      .count() \
      .orderBy("target") \
      .show()